# Model playground

This notebook is for testing out different models and architectures on a new dataset, [DermaMNIST](https://medmnist.com/): 28x28 colour dermatoscopic images of 7 types of skin lesions. We have loaded the dataset for you and defined a `DataLoader` for it.
The goal is to train a model that achieves at least 75% accuracy on the test set.

Note that the dataset is imbalanced: about 67% of the images are melanocytic nevi, so a model that always predicts this class already gets 67% accuracy. Also look at the per-class performance of your model (for example with a confusion matrix), not only at the accuracy.

You can use any model architecture, any optimizer, and any loss function.

We expect you to try out:
- Regularization techniques: dropout, weight decay, early stopping, data augmentation
- Transfer learning
- Hyperparameter tuning

We have provided you with some code to get started. Feel free to modify it. You can write your own architecture in `cm2003/models/custom_model.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload re-imports the files in cm2003/ when you change them, so you do not need to restart
# the kernel after editing train.py or the models.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as v2
import matplotlib.pyplot as plt
import numpy as np

from cm2003.train import train, check_accuracy
from cm2003.plotting import plot_results

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')  # Apple-silicon Mac
else:
    device = torch.device('cpu')
print('using device:', device)

In [ ]:
from medmnist import DermaMNIST

# Mean and standard deviation of each colour channel (R, G, B) of the DermaMNIST training images
MEAN, STD = (0.7631, 0.5381, 0.5614), (0.1366, 0.1543, 0.1692)

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

dataset_train = DermaMNIST(root='cm2003/datasets', split='train', transform=transform, download=True)
loader_train = DataLoader(dataset_train, batch_size=64, shuffle=True)

dataset_val = DermaMNIST(root='cm2003/datasets', split='val', transform=transform, download=True)
loader_val = DataLoader(dataset_val, batch_size=64, shuffle=False)

dataset_test = DermaMNIST(root='cm2003/datasets', split='test', transform=transform, download=True)
loader_test = DataLoader(dataset_test, batch_size=64, shuffle=False)

CLASS_NAMES = dataset_train.info['label']
print('Classes:', CLASS_NAMES)

In [ ]:
image, label = dataset_train[5]
print(f"Image shape: {image.shape}")

# Reshape and convert the image for display
image_display = image.permute(1, 2, 0).numpy()  # Change from (3, 28, 28) to (28, 28, 3)
image_display = image_display * np.array(STD) + np.array(MEAN)  # Undo the normalization
image_display = np.clip(image_display, 0, 1)  # Clip values to [0, 1] range

plt.imshow(image_display)
plt.title(f'Label: {label.item()} ({CLASS_NAMES[str(label.item())]})')
plt.axis('off')
plt.show()

In [ ]:
from cm2003.models import alexnet


model = alexnet.AlexNet(num_classes=7, in_channels=3)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.to(device)

# Set number of epochs
num_epochs = 10


# Train the model
model, results = train(model, loader_train, loader_val, criterion, optimizer, device, num_epochs)

# Check accuracy on the test set
test_loss, test_accuracy = check_accuracy(model, loader_test, criterion, device)
train_loss, train_accuracy = check_accuracy(model, loader_train, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

In [ ]:
plot_results(results, title='AlexNet on DermaMNIST')